# UrbanSound8K CNN Baseline on Google Colab

Purpose: run the CNN baseline training on Colab GPU while keeping GitHub as the source of truth for code.

Expected outputs:
- `results/cnn_baseline_fold10/`
- `figures/cnn_baseline_fold10_confusion_matrix.png`

Before running: open `Runtime > Change runtime type` and select a GPU accelerator.

In [ ]:
# Check the Colab GPU. This confirms that the runtime is using GPU acceleration.
!nvidia-smi

In [ ]:
# Clone the GitHub repository. GitHub should be treated as the source of truth.
# If the folder already exists, remove it first or restart the Colab runtime.
!git clone https://github.com/lezhe0414/urbansound8k-thesis.git
%cd urbansound8k-thesis

# Record the exact code version used for this experiment.
!git log --oneline -1

In [ ]:
# Install Python dependencies from the repository.
# This may take several minutes on a fresh Colab runtime.
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

In [ ]:
# Download and validate UrbanSound8K through soundata.
# Raw audio is not committed to Git because the dataset is large.
!python - <<'PY'
import soundata

dataset = soundata.initialize(
    "urbansound8k",
    data_home="data/raw/UrbanSound8K_soundata",
)
dataset.download()
dataset.validate()
PY

In [ ]:
# Convert audio clips into normalized Mel-spectrogram features.
# Output features are cached locally inside the Colab runtime.
!python -m src.preprocess \
  --raw-dir data/raw/UrbanSound8K_soundata \
  --out-dir data/processed/urbansound8k_mels

In [ ]:
# Train the CNN baseline using official UrbanSound8K fold 10 as the test fold.
# This is the main missing formal baseline result for the thesis comparison.
!python -m src.train --config configs/cnn_baseline.yaml --fold 10

In [ ]:
# Re-run evaluation from the saved checkpoint and regenerate final metrics/figures.
!python -m src.evaluate --run-dir results/cnn_baseline_fold10

In [ ]:
# Inspect the final metrics file.
# Copy these values back into docs/progress_tracker.md after downloading results.
!python - <<'PY'
import json
from pathlib import Path

metrics_path = Path("results/cnn_baseline_fold10/metrics.json")
with metrics_path.open("r", encoding="utf-8") as f:
    metrics = json.load(f)

for key, value in metrics.items():
    print(f"{key}: {value}")
PY

In [ ]:
# Package the CNN result artifacts for download back to the local machine.
# After downloading, place them in the same paths inside the local repository.
!zip -r cnn_baseline_fold10_artifacts.zip \
  results/cnn_baseline_fold10 \
  figures/cnn_baseline_fold10_confusion_matrix.png \
  figures/cnn_baseline_fold10_evaluation_confusion_matrix.png

## Local sync checklist

After Colab finishes:

1. Download `cnn_baseline_fold10_artifacts.zip`.
2. Unzip it into the local repository root.
3. Update `docs/progress_tracker.md` with the CNN metrics.
4. Run `python3 scripts/check_project_status.py` locally.
5. Commit only code/docs/config changes. Do not commit raw data or large generated artifacts unless explicitly required.